## Feature Engineering

Import the required libraries

In [58]:
import pandas as pd
pd.set_option('display.max_columns', 100)


Import the cleaned dataset

In [59]:
df = pd.read_csv('flights_eda_df.csv')
df.sample(3)

,origin,City,destination,City_destination,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,base_price,departure_hour,arrival_hour,departure_time_period,arrival_time_period,duration_group
9923,YOW,Ottawa,YVR,Vancouver,WestJet,7M8,2026-03-08,2026-06-18,07:45:00,Thursday,6,2026-06-18,17:00:00,102,370,1,122,7,17,Early Morning,Evening,"[360, 420)"
8222,YOW,Ottawa,YVR,Vancouver,WestJet,7M8,2026-03-08,2026-05-18,06:00:00,Monday,5,2026-05-18,14:01:00,71,351,1,122,6,14,Early Morning,Afternoon,"[300, 360)"
18293,YOW,Ottawa,YYC,Calgary,Air Canada,223,2026-03-08,2026-04-12,06:00:00,Sunday,4,2026-04-12,12:13:00,35,333,1,231,6,12,Early Morning,Afternoon,"[300, 360)"


In [60]:
df.shape

(43157, 22)

### I. Features to Create
- Weekend Flag: is_weekend = day_of_week_departure.isin(['Saturday', 'Sunday']).astype(int) (weekends often pricier).
- Season: season = month_departure.apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall') (seasonal demand).

In [61]:
# creata a weekend feature
df['is_weekend_departure'] = df['day_of_week_departure'].isin(['Saturday', 'Sunday']).astype(int)

# seasonal feature
df['season'] = df['month_departure'].apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall')

# days until departure group
# df['trip_departure_group'] = pd.cut(df['trip_duration_minutes'], 
# bins=[-1, 50, 100, 150, 200, 250, 300, 350, float('inf')], 
# labels=['0-50 days', '51-100 days', '101-150 days', '151-200 days', '200-250 days', '251-300 days', '301-350 days', '350+ days'])

# airline-route interaction feature
# df['airline_route'] = df['Name_airline'] + '_' + df['origin'] + '-' + df['destination']

# distanct bins feature
# df['distance_bin'] = pd.cut(df['distance_km'], 
# bins=[-1, 500, 1000, 1500, 2000, 2500, 3000, 3500, float('inf')], 
# labels=['0-500 km', '501-1000 km', '1001-1500 km', '1501-2000 km', '2001-2500 km', '2501-3000 km', '3001-3500 km', '3500+ km'])

### II. Features Selection
#### Drop
- Redundant/High Correlation: City, City_destination (use origin/destination codes instead). month_departure (seasonal info better binned).
- Irrelevant/Noise: departure_clock_time, arrival_clock_time (already binned into periods). base_price if predicting total_price (avoid leakage).
- Low Impact: Name_airline outliers like First Air (already dropped). Any constant columns (e.g., if all flights are domestic).
- Post-EDA Drops: Temporary columns like route.

#### Keep
- Numerical: distance_km, days_until_departure, trip_duration_minutes, number_of_stops, bookable_seats (all correlate with price; bookable_seats indicates demand)
- Categorical: origin, destination, Name_airline, aircraft, day_of_week_departure, departure_time_period, arrival_time_period (capture route/airline/time effects).
- Target variable: base_price

In [62]:
# Using .loc to select the relevant features for modeling
df2 = df.loc[:,['base_price','origin','destination', 'departure_hour', 'arrival_hour',
       'Name_airline', 'day_of_week_departure', 'days_until_departure',
       'trip_duration_minutes', 'number_of_stops','departure_time_period',
       'arrival_time_period', 'is_weekend_departure', 
       'season']]

df2.head(2)

,base_price,origin,destination,departure_hour,arrival_hour,Name_airline,day_of_week_departure,days_until_departure,trip_duration_minutes,number_of_stops,departure_time_period,arrival_time_period,is_weekend_departure,season
0,216,YOW,YYZ,5,7,WestJet,Monday,1,81,0,Early Morning,Early Morning,0,Spring
1,216,YOW,YYZ,6,8,Porter Airlines Canada Limited,Monday,1,75,0,Early Morning,Morning,0,Spring


### III. Dummy Variables

Use pandas .get_dummies to encode the categorical columns to numerical.

In [63]:
df3 = pd.get_dummies(df2, columns = ['origin', 'destination', 'Name_airline', 'day_of_week_departure',
       'departure_time_period', 'arrival_time_period', 'season',
       ], drop_first=True).astype(int)

df3.head(2)

,base_price,departure_hour,arrival_hour,days_until_departure,trip_duration_minutes,number_of_stops,is_weekend_departure,origin_YVR,origin_YYC,origin_YYZ,destination_YVR,destination_YYC,destination_YYZ,Name_airline_Air North Yukon's Airline,Name_airline_Air Transat,Name_airline_Central Mountain Air LTD,Name_airline_Pacific Coastal Airlines Limited,Name_airline_Porter Airlines Canada Limited,Name_airline_WestJet,day_of_week_departure_Monday,day_of_week_departure_Saturday,day_of_week_departure_Sunday,day_of_week_departure_Thursday,day_of_week_departure_Tuesday,day_of_week_departure_Wednesday,departure_time_period_Early Morning,departure_time_period_Evening,departure_time_period_Late Evening,departure_time_period_Morning,departure_time_period_Night,arrival_time_period_Early Morning,arrival_time_period_Evening,arrival_time_period_Late Evening,arrival_time_period_Morning,arrival_time_period_Night,season_Spring,season_Summer,season_Winter
0,216,5,7,1,81,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0
1,216,6,8,1,75,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0


In [64]:
# export csv
df3.to_csv('final_df.csv', index=False)
print('File saved')

File saved
